# Exponential Moving Average (EMA) - Complete Trading Tutorial

## Overview
This notebook is a comprehensive deep-dive into the Exponential Moving Average (EMA) including:
- Mathematical formulation & smoothing factor intuition
- Manual and vectorized implementations
- Strategy construction (crossover, price vs EMA, triple / ribbon)
- Advanced adaptive & volatility-aware variants
- Backtesting, optimization & walk-forward evaluation
- Risk management overlays and volume / multi-timeframe confirmation
- Practical exercises for further exploration

> Disclaimer: Educational purposes only. Not financial advice.

## 1. Import Required Libraries & Setup
We'll import required dependencies and configure visualization / display settings.

In [ ]:
# Section 1: Imports & Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings

# Optional libraries
try:
    import ccxt
    HAS_CCXT = True
except ImportError:
    HAS_CCXT = False

try:
    import talib
    HAS_TALIB = True
except ImportError:
    HAS_TALIB = False

try:
    import pandas_ta as ta
    HAS_PANDAS_TA = True
except ImportError:
    HAS_PANDAS_TA = False

# Project utilities
from utils.data_downloader import DataDownloader
from utils.indicators import IndicatorCalculator
from utils.backtest_engine import BacktestEngine, StrategyGenerator

# Plot styles
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

warnings.filterwarnings('ignore')
print('Environment configured:')
print(f'ccxt: {HAS_CCXT}, TA-Lib: {HAS_TALIB}, pandas-ta: {HAS_PANDAS_TA}')

## 2. EMA Formula & Parameter Explanation
The Exponential Moving Average applies exponentially decaying weights to past prices.

Mathematical recurrence:

$EMA_t = \alpha P_t + (1 - \alpha) EMA_{t-1}$  with  $\alpha = \frac{2}{n+1}$

Where:
- $P_t$ = price (typically close) at time t
- $n$ = lookback period (window length)
- $\alpha$ = smoothing factor (larger => more weight on recent data)

Properties:
- Effective lookback is roughly ~ 2n
- Weight of observation k periods back: $(1-\alpha)^k \alpha$
- Half-life (period at which weight decays to 0.5) = $\ln(0.5) / \ln(1-\alpha)$

Common periods and smoothing factors:

In [ ]:
# Helper: Smoothing factor table
from math import log

def ema_alpha(period: int) -> float:
    return 2 / (period + 1)

def ema_half_life(alpha: float) -> float:
    return log(0.5) / log(1 - alpha)

periods_example = [9, 12, 20, 26, 50, 100, 200]
rows = []
for p in periods_example:
    a = ema_alpha(p)
    hl = ema_half_life(a)
    rows.append({
        'period': p,
        'alpha': round(a, 6),
        'half_life_periods': round(hl, 2),
        'approx_effective_lookback(~2n)': 2*p
    })
alpha_table = pd.DataFrame(rows)
alpha_table

## 3. Data Download (Crypto OHLCV)
We pull daily BTC/USDT OHLCV for the last 365 days. Perform sanity checks: non-negative columns, monotonic datetime index, missing values.

In [ ]:
# Section 3: Data Download
pair = 'BTC/USDT'
timeframe = '1d'
days = 365

# Prefer downloader (online) else fallback to local CSV if exists
import os
from pathlib import Path

downloader = DataDownloader()
df = None
if HAS_CCXT:
    try:
        print(f'Downloading {pair} {timeframe} for {days} days via ccxt...')
        df = downloader.get_market_data(pair, timeframe, days, source='ccxt')
    except Exception as e:
        print('Download failed, will attempt local CSV fallback:', e)

# Local fallback path
csv_path = Path('notebooks/data/BTC_USDT_1d_365d.csv')
if (df is None or len(df)==0) and csv_path.exists():
    print('Loading local CSV fallback...')
    df = pd.read_csv(csv_path)
    # Ensure datetime index
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.set_index('timestamp')
    elif 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date')

# Basic validations
assert df is not None and len(df) > 0, 'No data available.'
assert df.index.is_monotonic_increasing, 'Index not monotonic increasing.'
for col in ['open','high','low','close','volume']:
    assert (df[col] >= 0).all(), f'Negative values found in {col}'

missing = df.isna().sum()
print('Data shape:', df.shape)
print('Date range:', df.index.min(), '->', df.index.max())
print('Missing values per column:')
print(missing[missing>0] if missing.sum()>0 else 'None')

display(df.head())
display(df.tail())

## 4. Manual EMA Calculation (Recursive Implementation)
We implement a manual EMA to understand the recursion and seeding choices.

Seeding options:
- `first`: EMA_0 = first closing price
- `sma`: EMA_{n-1} = SMA of first n closes, then start recursion

We validate resultant length and NaN handling.

In [ ]:
# Section 4: Manual EMA Implementation
from typing import Literal

def ema_manual(prices: pd.Series, period: int, seed_method: Literal['first','sma']='sma') -> pd.Series:
    assert period > 0, 'Period must be positive'
    prices = prices.astype(float)
    alpha = ema_alpha(period)
    ema_vals = []
    if seed_method == 'first':
        seed = prices.iloc[0]
        ema_vals.append(seed)
        start_idx = 1
    else:  # sma seed
        if len(prices) < period:
            raise ValueError('Not enough data for SMA seed')
        seed = prices.iloc[:period].mean()
        # fill leading NaNs to align length
        ema_vals = [np.nan]*(period-1) + [seed]
        start_idx = period
    prev = seed
    for i in range(start_idx, len(prices)):
        p = prices.iloc[i]
        val = alpha * p + (1 - alpha) * prev
        ema_vals.append(val)
        prev = val
    return pd.Series(ema_vals, index=prices.index, name=f'ema_{period}_manual')

# Compute example EMAs
ema_periods = [9,20,50,200]
manual_df = df.copy()
for p in ema_periods:
    manual_df[f'ema_{p}_manual'] = ema_manual(manual_df['close'], p, seed_method='sma')

print('Manual EMA columns added:')
print([c for c in manual_df.columns if 'ema_' in c and c.endswith('_manual')])
manual_df[[c for c in manual_df.columns if c.endswith('_manual')]].head()

## 5. Vectorized / Library EMA Implementations
We compute EMAs using pandas `ewm`, project `IndicatorCalculator`, `pandas_ta`, and `TA-Lib` (if available).

In [ ]:
# Section 5: Library EMA Implementations
lib_df = df.copy()
for p in ema_periods:
    lib_df[f'ema_{p}_pandas'] = lib_df['close'].ewm(span=p, adjust=False).mean()
    lib_df[f'ema_{p}_util'] = IndicatorCalculator.ema(lib_df['close'], p)
    if HAS_PANDAS_TA:
        lib_df.ta.ema(length=p, append=True)  # creates EMA_{p}
        lib_df.rename(columns={f'EMA_{p}': f'ema_{p}_ta'}, inplace=True)
    else:
        lib_df[f'ema_{p}_ta'] = np.nan
    if HAS_TALIB:
        lib_df[f'ema_{p}_talib'] = talib.EMA(lib_df['close'].values, timeperiod=p)
    else:
        lib_df[f'ema_{p}_talib'] = np.nan

cols_preview = [c for c in lib_df.columns if any(c.startswith(f'ema_{p}_') for p in ema_periods)][:12]
lib_df[cols_preview].head()

## 6. Implementation Comparison & Numerical Verification
We compare manual EMA vs pandas, utils, pandas-ta, TA-Lib (period=20). We check max/mean absolute differences and validate tolerance < 1e-8 (where libraries present).

In [ ]:
# Section 6: Comparison for period=20
compare_period = 20
cmp = pd.DataFrame({
    'manual': manual_df[f'ema_{compare_period}_manual'],
    'pandas': lib_df[f'ema_{compare_period}_pandas'],
    'util': lib_df[f'ema_{compare_period}_util'],
    'pandas_ta': lib_df[f'ema_{compare_period}_ta'],
    'talib': lib_df[f'ema_{compare_period}_talib']
})

# Drop initial NaNs for fair comparison
cmp_clean = cmp.dropna()

metrics = {}
for col in cmp_clean.columns:
    if col == 'manual':
        continue
    diff = (cmp_clean['manual'] - cmp_clean[col]).abs()
    metrics[col] = {
        'max_abs_diff': diff.max(),
        'mean_abs_diff': diff.mean()
    }

metrics_df = pd.DataFrame(metrics).T
print('Difference metrics vs manual (period=20):')
print(metrics_df)

# Assert tolerance for available implementations (skip NaN columns entirely)
for col in ['pandas','util']:
    assert metrics_df.loc[col,'max_abs_diff'] < 1e-8, f'{col} diff too large'

print('\nManual last 5-step expansion (period=20):')
# Reconstruct last 5 recursive steps
alpha = ema_alpha(compare_period)
last_prices = df['close'].iloc[-5:]
last_manual_values = manual_df[f'ema_{compare_period}_manual'].iloc[-6:]
print(f'alpha={alpha:.6f}')
for i in range(1,6):
    p = last_prices.iloc[i-1]
    prev = last_manual_values.iloc[i-1]
    theoretical = alpha * p + (1-alpha)*prev
    actual = manual_df[f'ema_{compare_period}_manual'].iloc[-6+i]
    print(f'Step {i}: price={p:.2f} prevEMA={prev:.2f} theoretical={theoretical:.6f} actual={actual:.6f} diff={abs(theoretical-actual):.6e}')

cmp.tail()

## 7. Visualization: Price with Multiple EMAs (9, 20, 50, 200)
We plot candlesticks with overlays. Background shading indicates fast (9) > slow (20) regime.

In [ ]:
# Section 7: Plot EMAs
viz_df = manual_df.copy()
# Ensure we have consistent EMAs from manual; optionally could use pandas versions
fast = viz_df['ema_9_manual']
mid = viz_df['ema_20_manual']
slow = viz_df['ema_50_manual']
base = viz_df['ema_200_manual']

fig = make_subplots(rows=1, cols=1, subplot_titles=(f'{pair} - EMA Stack',))
fig.add_trace(go.Candlestick(x=viz_df.index, open=viz_df['open'], high=viz_df['high'], low=viz_df['low'], close=viz_df['close'], name='Price'), row=1, col=1)
for series, name, color in [
    (fast, 'EMA(9)', 'orange'),
    (mid, 'EMA(20)', 'blue'),
    (slow, 'EMA(50)', 'red'),
    (base, 'EMA(200)', 'purple')
]:
    fig.add_trace(go.Scatter(x=viz_df.index, y=series, mode='lines', name=name, line=dict(width=2, color=color)), row=1, col=1)

# Background shading where fast > mid
regime = (fast > mid).astype(int)
# Identify regime change points
change_idx = regime.diff().fillna(0) != 0
segments = []
current_start = None
current_state = None
for ts, val in regime.items():
    if current_state is None:
        current_state = val
        current_start = ts
    elif val != current_state:
        segments.append((current_start, ts, current_state))
        current_start = ts
        current_state = val
segments.append((current_start, viz_df.index[-1], current_state))

for start, end, state in segments:
    if state == 1:
        fig.add_vrect(x0=start, x1=end, fillcolor='rgba(50,200,50,0.08)', line_width=0)

fig.update_layout(height=600, template='plotly_dark', xaxis_rangeslider_visible=False, title='EMAs with Regime Shading (Fast > Mid)')
fig.show()

# Static plot
plt.figure(figsize=(15,7))
plt.plot(viz_df.index, viz_df['close'], label='Close', color='black', alpha=0.6)
plt.plot(viz_df.index, fast, label='EMA(9)', color='orange')
plt.plot(viz_df.index, mid, label='EMA(20)', color='blue')
plt.plot(viz_df.index, slow, label='EMA(50)', color='red')
plt.plot(viz_df.index, base, label='EMA(200)', color='purple')
plt.title(f'{pair} - EMAs')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. EMA Crossover Strategy (Short vs Long)
Classic approach: long when fast EMA crosses above slow EMA; short (or flat) when crosses below. Default (12,26) mirrors MACD base EMAs.

In [ ]:
# Section 8: EMA Crossover Strategy

def ema_crossover_strategy(data: pd.DataFrame, short=12, long=26) -> pd.DataFrame:
    d = data.copy()
    d[f'ema_{short}'] = IndicatorCalculator.ema(d['close'], short)
    d[f'ema_{long}'] = IndicatorCalculator.ema(d['close'], long)
    d['signal'] = 0
    d.loc[d[f'ema_{short}'] > d[f'ema_{long}'], 'signal'] = 1
    d.loc[d[f'ema_{short}'] < d[f'ema_{long}'], 'signal'] = -1
    d['crossover'] = (d[f'ema_{short}'] > d[f'ema_{long}']) & (d[f'ema_{short}'].shift(1) <= d[f'ema_{long}'].shift(1))
    d['crossunder'] = (d[f'ema_{short}'] < d[f'ema_{long}']) & (d[f'ema_{short}'].shift(1) >= d[f'ema_{long}'].shift(1))
    return d

ema_cross_df = ema_crossover_strategy(df, 12, 26)
print('Crossovers:', ema_cross_df['crossover'].sum(), 'Crossunders:', ema_cross_df['crossunder'].sum())

fig = make_subplots(rows=1, cols=1, subplot_titles=('EMA Crossover 12/26',))
fig.add_trace(go.Candlestick(x=ema_cross_df.index, open=ema_cross_df['open'], high=ema_cross_df['high'], low=ema_cross_df['low'], close=ema_cross_df['close'], name='Price'), row=1,col=1)
fig.add_trace(go.Scatter(x=ema_cross_df.index, y=ema_cross_df['ema_12'], mode='lines', name='EMA(12)', line=dict(color='orange')), row=1,col=1)
fig.add_trace(go.Scatter(x=ema_cross_df.index, y=ema_cross_df['ema_26'], mode='lines', name='EMA(26)', line=dict(color='blue')), row=1,col=1)

buys = ema_cross_df[ema_cross_df['crossover']]
fig.add_trace(go.Scatter(x=buys.index, y=buys['close'], mode='markers', name='Buy', marker=dict(symbol='triangle-up', size=12, color='green')), row=1,col=1)

sells = ema_cross_df[ema_cross_df['crossunder']]
fig.add_trace(go.Scatter(x=sells.index, y=sells['close'], mode='markers', name='Sell', marker=dict(symbol='triangle-down', size=12, color='red')), row=1,col=1)

fig.update_layout(height=550, template='plotly_dark', xaxis_rangeslider_visible=False)
fig.show()

## 9. Price vs Single EMA Strategy
Signal when price closes above/below mid-term EMA(50).

In [ ]:
# Section 9: Price vs EMA(50)

def price_vs_ema_strategy(data: pd.DataFrame, period=50) -> pd.DataFrame:
    d = data.copy()
    d[f'ema_{period}'] = IndicatorCalculator.ema(d['close'], period)
    d['signal'] = 0
    d.loc[d['close'] > d[f'ema_{period}'], 'signal'] = 1
    d.loc[d['close'] < d[f'ema_{period}'], 'signal'] = -1
    d['price_above_event'] = (d['close'] > d[f'ema_{period}']) & (d['close'].shift(1) <= d[f'ema_{period}'].shift(1))
    d['price_below_event'] = (d['close'] < d[f'ema_{period}']) & (d['close'].shift(1) >= d[f'ema_{period}'].shift(1))
    return d

price_ema_df = price_vs_ema_strategy(df, 50)
print('Above events:', price_ema_df['price_above_event'].sum(), 'Below events:', price_ema_df['price_below_event'].sum())
price_ema_df[['close','ema_50','price_above_event','price_below_event']].tail()

## 10. Triple EMA / EMA Ribbon Strategy
We use (9, 21, 55) for strong trend detection and optionally extend to a ribbon.

In [ ]:
# Section 10: Triple EMA and Ribbon

def triple_ema_strategy(data: pd.DataFrame, a=9,b=21,c=55) -> pd.DataFrame:
    d = data.copy()
    d[f'ema_{a}'] = IndicatorCalculator.ema(d['close'], a)
    d[f'ema_{b}'] = IndicatorCalculator.ema(d['close'], b)
    d[f'ema_{c}'] = IndicatorCalculator.ema(d['close'], c)
    d['strong_up'] = (d[f'ema_{a}'] > d[f'ema_{b}']) & (d[f'ema_{b}'] > d[f'ema_{c}'])
    d['strong_down'] = (d[f'ema_{a}'] < d[f'ema_{b}']) & (d[f'ema_{b}'] < d[f'ema_{c}'])
    d['signal'] = 0
    d.loc[d['strong_up'], 'signal'] = 1
    d.loc[d['strong_down'], 'signal'] = -1
    d['up_start'] = d['strong_up'] & ~d['strong_up'].shift(1).fillna(False)
    d['down_start'] = d['strong_down'] & ~d['strong_down'].shift(1).fillna(False)
    return d

triple_df = triple_ema_strategy(df)

# Ribbon extension
ribbon_periods = [8,13,21,34,55,89]
ribbon_df = df.copy()
for rp in ribbon_periods:
    ribbon_df[f'ema_{rp}'] = IndicatorCalculator.ema(ribbon_df['close'], rp)

# Bullish ordering count (strictly ascending by period)
# Compute rank order each row
ema_cols = [f'ema_{rp}' for rp in ribbon_periods]
order_matrix = ribbon_df[ema_cols].copy()
# Count monotonic increasing (shortest period highest or lowest?) For bullish: shorter period EMAs above longer ones
bullish = np.ones(len(order_matrix), dtype=bool)
for i in range(len(ribbon_periods)-1):
    bullish &= order_matrix[ema_cols[i]] > order_matrix[ema_cols[i+1]]
ribbon_df['bullish_ribbon'] = bullish
ribbon_df['bullish_density'] = ribbon_df[ema_cols].rank(axis=1, method='first').apply(lambda r: (r.diff() < 0).sum(), axis=1)
print('Triple strategy up starts:', triple_df['up_start'].sum(), 'down starts:', triple_df['down_start'].sum())
triple_df[['close','ema_9','ema_21','ema_55','strong_up','strong_down']].tail()

## 11. Signal Encoding & Consolidation for Backtesting
We standardize each strategy's signals into aligned DataFrames (no forward leakage).

In [ ]:
# Section 11: Consolidate signals
signals_dict = {
    'EMA_Crossover_12_26': ema_cross_df[['signal']].copy(),
    'Price_vs_EMA50': price_ema_df[['signal']].copy(),
    'Triple_EMA_9_21_55': triple_df[['signal']].copy()
}
# Ensure alignment & no lookahead (position uses previous day's signal in engine)
for name, s in signals_dict.items():
    s['signal'] = s['signal'].fillna(0)

list(signals_dict.keys())

## 12. Backtesting Engine Execution (All EMA Strategies)
Run the engine for each strategy and collect metrics.

In [ ]:
# Section 12: Run Backtests
engine = BacktestEngine(initial_capital=10000)
for name, sigs in signals_dict.items():
    print('Running backtest:', name)
    engine.run_backtest(df, sigs, name)

summary = engine.compare_strategies(list(signals_dict.keys()))
summary.sort_values('sharpe_ratio', ascending=False)

## 13. Equity Curve, Drawdown & Metrics Visualization
Overlay equity curves and drawdowns; bar chart for key metrics.

In [ ]:
# Section 13: Visualization
engine.plot_equity_curve(list(signals_dict.keys()))
engine.plot_drawdown(list(signals_dict.keys()))

# Bar chart metrics
metric_plot = summary[['annualized_return','sharpe_ratio','max_drawdown']].copy()
fig, axes = plt.subplots(1,3, figsize=(15,4))
metric_plot['annualized_return'].plot(kind='bar', ax=axes[0], title='Annualized Return')
metric_plot['sharpe_ratio'].plot(kind='bar', ax=axes[1], title='Sharpe')
metric_plot['max_drawdown'].plot(kind='bar', ax=axes[2], title='Max Drawdown')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
summary

## 14. Trade Extraction & Performance Statistics
Convert position changes to discrete trades for the best Sharpe strategy.

In [ ]:
# Section 14: Trade extraction
best = summary['sharpe_ratio'].idxmax()
print('Best strategy by Sharpe:', best)
res = engine.results[best]['data']
trades = []
position = 0
entry_price = None
entry_date = None
for date, row in res.iterrows():
    sig = row['signal']
    if position == 0 and sig != 0:
        position = sig
        entry_price = row['close']
        entry_date = date
    elif position != 0 and sig == 0:
        exit_price = row['close']
        pnl = (exit_price - entry_price) if position == 1 else (entry_price - exit_price)
        pct = pnl / entry_price
        trades.append({'entry_date': entry_date, 'exit_date': date, 'position':'Long' if position==1 else 'Short', 'entry_price': entry_price, 'exit_price': exit_price, 'profit_pct': pct, 'hold_days': (date-entry_date).days})
        position = 0
        entry_price=None
        entry_date=None
trades_df = pd.DataFrame(trades)
if len(trades_df):
    win = trades_df[trades_df['profit_pct']>0]
    loss = trades_df[trades_df['profit_pct']<=0]
    expectancy = win['profit_pct'].mean()*len(win)/len(trades_df) + loss['profit_pct'].mean()*len(loss)/len(trades_df) if len(loss) else win['profit_pct'].mean()
    profit_factor = win['profit_pct'].sum()/abs(loss['profit_pct'].sum()) if len(loss) and abs(loss['profit_pct'].sum())>0 else np.nan
    print(f"Trades: {len(trades_df)} Win rate: {len(win)/len(trades_df):.2%} Avg win: {win['profit_pct'].mean():.2%} Avg loss: {loss['profit_pct'].mean():.2% if len(loss) else 'N/A'}")
    print(f"Expectancy: {expectancy:.2%} Profit factor: {profit_factor:.2f if profit_factor==profit_factor else 'N/A'}")
    display(trades_df.tail())
else:
    print('No completed trades captured (signals may not revert to flat).')

## 15. Advanced EMA Techniques (Ribbon, Distance, Slope, Acceleration)
Compute ribbon, distance %, slope (ema20 diff), and acceleration (second diff).

In [ ]:
# Section 15: Advanced metrics
adv = df.copy()
adv['ema_20'] = IndicatorCalculator.ema(adv['close'], 20)
adv['ema_50'] = IndicatorCalculator.ema(adv['close'], 50)
adv['distance_pct'] = (adv['close'] - adv['ema_50']) / adv['ema_50'] * 100
adv['slope_ema20'] = adv['ema_20'].diff()
adv['accel_ema20'] = adv['slope_ema20'].diff()

# Ribbon heatmap (period ordering) reuse ribbon_df from earlier
heat = ribbon_df[[col for col in ribbon_df.columns if col.startswith('ema_') and any(col.endswith(str(p)) for p in [8,13,21,34,55,89])]].copy()
# Compute ordering matrix rank normalized 0..1
ranked = heat.rank(axis=1, method='dense') / len(heat.columns)

plt.figure(figsize=(12,4))
plt.plot(adv.index, adv['distance_pct'], label='Distance % (close vs EMA50)')
plt.axhline(0, color='black', linewidth=1)
plt.title('Distance % from EMA50')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,4))
plt.plot(adv.index, adv['slope_ema20'], label='EMA20 Slope')
plt.plot(adv.index, adv['accel_ema20'], label='EMA20 Acceleration', alpha=0.6)
plt.axhline(0, color='black', linewidth=1)
plt.legend(); plt.title('EMA20 Slope & Acceleration'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print('Distance stats (%): min={:.2f} max={:.2f} mean={:.2f}'.format(adv['distance_pct'].min(), adv['distance_pct'].max(), adv['distance_pct'].mean()))

## 16. Volatility-Adjusted EMA (Adaptive Alpha)
We scale alpha by relative ATR to close (clipped) to create an adaptive EMA.

In [ ]:
# Section 16: Adaptive EMA
adapt = df.copy()
adapt['atr_14'] = IndicatorCalculator.atr(adapt['high'], adapt['low'], adapt['close'], 14)
base_period = 20
base_alpha = ema_alpha(base_period)
# Relative volatility ratio
ratio = (adapt['atr_14'] / adapt['close']).clip(lower=0.0005, upper=0.02)
# Scale alpha within bounds
adapt['alpha_t'] = (ratio / ratio.median()) * base_alpha
adapt['alpha_t'] = adapt['alpha_t'].clip(lower=base_alpha*0.5, upper=base_alpha*1.5)

vals = []
prev = adapt['close'].iloc[0]
for i,(price,a) in enumerate(zip(adapt['close'], adapt['alpha_t'])):
    if i==0:
        vals.append(price)
    else:
        vals.append(a*price + (1-a)*prev)
        prev = vals[-1]
adapt['adaptive_ema_20'] = vals
adapt['fixed_ema_20'] = IndicatorCalculator.ema(adapt['close'], base_period)

plt.figure(figsize=(14,5))
plt.plot(adapt.index, adapt['close'], label='Close', alpha=0.4)
plt.plot(adapt.index, adapt['fixed_ema_20'], label='EMA20 Fixed')
plt.plot(adapt.index, adapt['adaptive_ema_20'], label='Adaptive EMA20')
plt.legend(); plt.title('Adaptive vs Fixed EMA20'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print('Mean absolute difference:', (adapt['adaptive_ema_20']-adapt['fixed_ema_20']).abs().mean())

## 17. Risk Management with EMA (ATR Stops, Position Sizing, R:R)
Derive ATR-based stops, dynamic sizing using slope percentiles, and reward: risk analysis.

In [ ]:
# Section 17: Risk management calculations
risk = df.copy()
risk['ema_20'] = IndicatorCalculator.ema(risk['close'], 20)
risk['atr_14'] = IndicatorCalculator.atr(risk['high'], risk['low'], risk['close'], 14)
mult = 1.5
risk['stop_long'] = risk['ema_20'] - mult*risk['atr_14']
risk['stop_short'] = risk['ema_20'] + mult*risk['atr_14']
# Slope for sizing
risk['slope'] = risk['ema_20'].diff()
q30, q70 = risk['slope'].quantile([0.3,0.7])
conditions = [risk['slope'] < q30, risk['slope'].between(q30,q70), risk['slope'] > q70]
choices = [0.25, 0.5, 1.0]
risk['position_size'] = np.select(conditions, choices, default=0.25)
# Assume potential entry at close for demonstration
risk['risk_per_unit'] = risk['close'] - risk['stop_long']
risk['reward_target'] = risk['close'] + 2*risk['risk_per_unit']
risk['risk_reward_ratio'] = (risk['reward_target'] - risk['close']) / risk['risk_per_unit']
print('Avg pos size:', risk['position_size'].mean(), 'Avg R:R:', risk['risk_reward_ratio'].mean())

plt.figure(figsize=(12,5))
plt.plot(risk.index, risk['position_size'], label='Position Size Multiplier')
plt.title('Dynamic Position Sizing (EMA Slope Based)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

risk[['close','ema_20','stop_long','stop_short','position_size','risk_reward_ratio']].tail()

## 18. Parameter Optimization (Grid Search)
Search over short vs long EMA pairs selecting by Sharpe then lower drawdown.

In [ ]:
# Section 18: Grid search
short_periods = [5,8,10,12,14]
long_periods = [20,26,30,35,40]
results_rows = []
opt_engine = BacktestEngine(initial_capital=10000)
for sp in short_periods:
    for lp in long_periods:
        if sp >= lp: continue
        strat = ema_crossover_strategy(df, sp, lp)[['signal']]
        opt_engine.run_backtest(df, strat, f'opt_{sp}_{lp}')
        m = opt_engine.results[f'opt_{sp}_{lp}']['metrics']
        results_rows.append({'short': sp,'long': lp, **m})
opt_df = pd.DataFrame(results_rows)
opt_df = opt_df.sort_values(['sharpe_ratio','max_drawdown'], ascending=[False, True])
print('Top parameter sets:')
opt_df.head(10)

## 19. Walk-Forward Optimization (Rolling Windows)
Optimize on 180-day window; apply best to following 60 days; slide forward.

In [ ]:
# Section 19: Walk-forward
wf_short = short_periods
wf_long = long_periods
train_win = 180
test_win = 60
wf_records = []
wf_equity = []
wf_engine = BacktestEngine(initial_capital=1.0)
prices = df.copy()
start = 0
while start + train_win + test_win <= len(prices):
    train_slice = prices.iloc[start:start+train_win]
    test_slice = prices.iloc[start+train_win:start+train_win+test_win]
    # Optimize on train
    best_params = None
    best_sharpe = -1e9
    best_dd = 1
    for sp in wf_short:
        for lp in wf_long:
            if sp >= lp: continue
            strat = ema_crossover_strategy(train_slice, sp, lp)[['signal']]
            tmp_engine = BacktestEngine(initial_capital=1.0)
            tmp_engine.run_backtest(train_slice, strat, 'tmp')
            met = tmp_engine.results['tmp']['metrics']
            sharpe = met['sharpe_ratio']
            dd = abs(met['max_drawdown'])
            if sharpe > best_sharpe or (abs(sharpe-best_sharpe)<1e-9 and dd < best_dd):
                best_params = (sp, lp)
                best_sharpe = sharpe
                best_dd = dd
    # Apply on test
    sp, lp = best_params
    test_strat = ema_crossover_strategy(test_slice, sp, lp)[['signal']]
    wf_engine.run_backtest(test_slice, test_strat, f'{start}')
    equity_part = wf_engine.results[f'{start}']['data']['cumulative_returns']
    wf_equity.append(equity_part)
    wf_records.append({'start_index': start, 'train_start': train_slice.index[0], 'train_end': train_slice.index[-1], 'test_end': test_slice.index[-1], 'short': sp, 'long': lp, 'train_sharpe': best_sharpe, 'train_dd': -best_dd})
    start += test_win
wf_df = pd.DataFrame(wf_records)
wf_equity_series = pd.concat(wf_equity)
plt.figure(figsize=(12,5))
plt.plot(wf_equity_series.index, wf_equity_series.values, label='Walk-Forward OOS Equity')
plt.title('Walk-Forward Out-of-Sample Equity (Normalized)')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()
wf_df.head()

## 20. Volume-Confirmed EMA Crossover Variant
Filter crossovers requiring volume above its EMA(20).

In [ ]:
# Section 20: Volume confirmation
vol_df = ema_crossover_strategy(df, 12, 26)
vol_df['vol_ema_20'] = IndicatorCalculator.ema(vol_df['volume'], 20)
vol_df['volume_condition'] = vol_df['volume'] > vol_df['vol_ema_20']
vol_df['signal_filtered'] = 0
vol_df.loc[vol_df['crossover'] & vol_df['volume_condition'], 'signal_filtered'] = 1
vol_df.loc[vol_df['crossunder'] & vol_df['volume_condition'], 'signal_filtered'] = -1

vol_engine = BacktestEngine(initial_capital=10000)
vol_engine.run_backtest(df, vol_df[['signal']], 'Unfiltered_12_26')
vol_engine.run_backtest(df, vol_df[['signal_filtered']].rename(columns={'signal_filtered':'signal'}), 'Volume_Filtered_12_26')
vol_summary = vol_engine.compare_strategies(['Unfiltered_12_26','Volume_Filtered_12_26'])
print('Trades removed by volume filter (crossover events):', (vol_df['crossover'].sum()+vol_df['crossunder'].sum()) - ( (vol_df['crossover'] & vol_df['volume_condition']).sum() + (vol_df['crossunder'] & vol_df['volume_condition']).sum()))
vol_summary

## 21. Multi-Timeframe EMA Alignment (Daily vs 4H)
Demonstration with synthetic 4H resample (if actual 4H download unavailable).

In [ ]:
# Section 21: Multi-Timeframe alignment (synthetic 4H from daily by interpolation)
# In production you'd download 4h data. Here we create a simple expanded index.
mtf_daily = df.copy()
# Create synthetic 4H index (6 periods per day) by forward filling
expanded = mtf_daily.reindex(pd.date_range(mtf_daily.index.min(), mtf_daily.index.max(), freq='4H'), method='ffill')
expanded['ema_fast_4h'] = IndicatorCalculator.ema(expanded['close'], 12)
expanded['ema_slow_4h'] = IndicatorCalculator.ema(expanded['close'], 26)
# Resample back to daily alignment by last 4H of day
fourh_daily = expanded[['ema_fast_4h','ema_slow_4h']].resample('1D').last()

mtf_daily['ema_fast_d'] = IndicatorCalculator.ema(mtf_daily['close'], 12)
mtf_daily['ema_slow_d'] = IndicatorCalculator.ema(mtf_daily['close'], 26)

aligned = mtf_daily.join(fourh_daily, how='inner')
aligned['daily_bull'] = aligned['ema_fast_d'] > aligned['ema_slow_d']
aligned['fourh_bull'] = aligned['ema_fast_4h'] > aligned['ema_slow_4h']
aligned['combined_signal'] = 0
aligned.loc[aligned['daily_bull'] & aligned['fourh_bull'], 'combined_signal'] = 1
aligned.loc[(~aligned['daily_bull']) & (~aligned['fourh_bull']), 'combined_signal'] = -1

mtf_engine = BacktestEngine(initial_capital=10000)
mtf_engine.run_backtest(aligned, aligned[['combined_signal']].rename(columns={'combined_signal':'signal'}), 'MTF_12_26')
mtf_engine.results['MTF_12_26']['metrics']

## 22. Exercises (Extensions & Challenges)
1. Implement Double EMA (DEMA) and compare lag vs standard EMA.
2. Add a Chandelier Exit around adaptive EMA entries.
3. Optimize scaling factors for adaptive alpha (grid search volatility scaling).
4. Create probabilistic signal strength using logistic(distance_pct) mapping.
5. Integrate realistic transaction costs & slippage; re-run Sharpe sensitivity.

---
### Bonus Ideas
- Add KAMA (Kaufman Adaptive Moving Average) comparison.
- Build a combined regime filter using ribbon density + volatility regime.
- Deploy signals to TradingView via alerts (outside Python scope).

Happy experimenting!